# Silver — Chamados (`case.silver.tb_chamados`)

Notebook de transformação da camada **Bronze → Silver** da tabela de chamados. O objetivo é entregar uma tabela limpa, deduplicada e com domínios de valores padronizados, pronta para consumo na camada Gold. Ao final, células de debug validam os resultados.

## Setup

Este bloco garante que o schema `case.silver` exista no Unity Catalog e define as variáveis de caminho (`bronze_schema` e `silver_schema`) usadas em todo o notebook.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "case"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

print(f"Schema {silver_schema} garantido. Iniciando transformações...")

## Transformação Bronze → Silver

Este bloco lê a `case.bronze.tb_chamados` e aplica as regras de tratamento antes de gravar a tabela Silver:

* **Dedup**: remove duplicidades por `id_chamado` (134.474 → 131.838 linhas)
* **IDs sentinelas**: `id_atendente` 0, 99 e 999 (sem cadastro na base de atendentes) viram `-1`, representando "chamado sem atendente"
* **Categóricas** (`motivo`, `canal`): minúsculas, acentuação unificada (para fundir grafias como `Cartão`/`cartao`), `_` vira espaço, `.` entre letras removido (`u.r.a` → `ura`), abreviações expandidas (`Atend.` → `atendimento`), `chat bot` → `chatbot` e espaços extras colapsados
* **`resolvido`**: padronizado em apenas `sim` / `nao` (variantes como `S`, `N`, `1`, `0`, `nao resolvido` são mapeadas); valores não reconhecidos e nulls permanecem null
* **Colunas de tempo**: as colunas de hora fixa (`hora_abertura_chamado`, `hora_inicio_atendimento`, `hora_finalizacao_atendimento`) são descartadas por serem 100% nulas ou sem significado
* **Durações**: `tempo_espera` e `tempo_atendimento` viram inteiros; só valores numéricos positivos são mantidos, todo o resto ("NULL", textos, zero, negativos) vira null

A gravação usa `overwrite` com `overwriteSchema`, já que o schema muda entre execuções (colunas dropadas e tipos alterados de string para int).

In [0]:
df_bronze_chamados = spark.table(f"{bronze_schema}.tb_chamados")

# Dedup por id_chamado 
df_silver_chamados = df_bronze_chamados.dropDuplicates(["id_chamado"])

# id_atendente: IDs sentinelas sem cadastro (0, 99, 999) -> -1 (chamado sem atendente)
df_silver_chamados = df_silver_chamados.withColumn(
    "id_atendente",
    F.when(F.col("id_atendente").isin(0, 99, 999), F.lit(-1)).otherwise(F.col("id_atendente"))
)

# === Tratamento das colunas categóricas ===
# Minúsculas, sem abreviações (Atend. -> Atendimento), sem "_", sem "." entre letras (u.r.a -> ura),
# "chat bot" -> "chatbot", acentuação unificada (cartao == cartão) e espaços extras colapsados
acentos    = "áàâãäéèêëíìîïóòôõöúùûüçñ"
sem_acento = "aaaaaeeeeiiiiooooouuuucn"

def normalizar_coluna(col_expr):
    c = F.translate(F.lower(F.trim(col_expr)), acentos, sem_acento)
    
    # Atendimento_Inicial -> Atendimento Inicial
    c = F.regexp_replace(c, "_", " ")                             
    
    # u.r.a -> ura
    c = F.regexp_replace(c, r"\.", "")                             
    
    # colapsa espaços múltiplos
    c = F.regexp_replace(c, " +", " ")                             
    
    # Atend. -> Atendimento (sem abreviação)
    c = F.regexp_replace(c, r"^atend(?!imento)", "atendimento")    
    
    # chat bot -> chatbot
    c = F.regexp_replace(c, "chat bot", "chatbot")                 
    return c

for coluna in ["motivo", "canal"]:
    df_silver_chamados = df_silver_chamados.withColumn(coluna, normalizar_coluna(F.col(coluna)))

# resolvido: manter apenas "sim" e "nao" (nulls continuam null)
resolvido_min = F.lower(F.trim(F.col("resolvido")))
resolvido_sem_acento = F.translate(resolvido_min, "ã", "a")
df_silver_chamados = df_silver_chamados.withColumn(
    "resolvido",
    F.when(resolvido_min.isin("sim", "s", "1"), F.lit("sim"))
     .when(resolvido_sem_acento.isin("nao", "n", "0", "nao resolvido"), F.lit("nao"))
     .otherwise(F.lit(None).cast("string"))
)

# Colunas de tempo: descartar as horas fixas (100% nulas ou sem significado)
df_silver_chamados = df_silver_chamados.drop(
    "hora_abertura_chamado", "hora_inicio_atendimento", "hora_finalizacao_atendimento"
)

# Durações: manter apenas números positivos como inteiros, todo o resto vira null
# (regex garante que só strings numéricas positivas sejam castadas; "NULL", 0 e negativos ficam null)
for coluna in ["tempo_espera", "tempo_atendimento"]:
    df_silver_chamados = df_silver_chamados.withColumn(
        coluna,
        F.when(F.col(coluna).rlike(r"^[1-9][0-9]*$"), F.col(coluna).cast("int"))
         .otherwise(F.lit(None).cast("int"))
    )

df_silver_chamados.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_chamados")

print(f"Tabela {silver_schema}.tb_chamados criada/atualizada com sucesso!")
print(f"Bronze (antes do dedup): {df_bronze_chamados.count():,} linhas")
print(f"Silver (após dedup e tratamento): {df_silver_chamados.count():,} linhas")

## Debug — Valores distintos por coluna

Validação pós-transformação: lista os valores distintos das colunas categóricas (`motivo` com 14 categorias, `canal` com 4, `resolvido` com apenas `sim`/`nao`/null), a contagem de IDs distintos (`id_atendente` com 26 valores: 25 atendentes reais + o `-1`) e amostras das colunas de tempo. Serve como conferência rápida de que as regras de padronização foram aplicadas.

In [0]:
# === DEBUG: Valores distintos por coluna ===
df_silver = spark.table(f"{silver_schema}.tb_chamados")

colunas = df_silver.columns

colunas_categoricas = ["motivo", "canal", "resolvido"]
colunas_ids = ["id_chamado", "id_cliente", "id_atendente"]
colunas_tempo = ["hora_abertura_chamado", "hora_inicio_atendimento", "hora_finalizacao_atendimento", "tempo_espera", "tempo_atendimento"]

print("=" * 60)
print("COLUNAS CATEGÓRICAS - Valores distintos")
print("=" * 60)
for col in colunas_categoricas:
    if col in colunas:
        distintos = df_silver.select(col).distinct().orderBy(col).collect()
        print(f"\n--- {col} ({len(distintos)} valores distintos) ---")
        for row in distintos:
            print(f"  {row[col]}")

print("\n" + "=" * 60)
print("COLUNAS DE ID - Contagem de distintos e amostra")
print("=" * 60)
for col in colunas_ids:
    if col in colunas:
        qtd = df_silver.select(col).distinct().count()
        amostra = [r[col] for r in df_silver.select(col).distinct().orderBy(col).limit(10).collect()]
        print(f"\n--- {col} ({qtd} valores distintos) ---")
        print(f"  Amostra (até 10): {amostra}")

print("\n" + "=" * 60)
print("COLUNAS DE TEMPO - Amostra de valores distintos (até 15)")
print("=" * 60)
for col in colunas_tempo:
    if col in colunas:
        qtd = df_silver.select(col).distinct().count()
        amostra = [r[col] for r in df_silver.select(col).distinct().orderBy(col).limit(15).collect()]
        print(f"\n--- {col} ({qtd} valores distintos) ---")
        for v in amostra:
            print(f"  {v}")

print("\n" + "=" * 60)
print(f"ingestion_datetime - min: {df_silver.agg(F.min('ingestion_datetime')).collect()[0][0]}")
print(f"ingestion_datetime - max: {df_silver.agg(F.max('ingestion_datetime')).collect()[0][0]}")
print("=" * 60)

## Debug — Qualidade e integridade referencial

Avaliação de qualidade restante na tabela Silver:

* **Nulos por coluna**: os nulos de `tempo_espera` (75.588) concentram-se 100% nos canais `ura` e `chatbot` — chamados sem fila de espera humana, comportamento estrutural e esperado. `tempo_atendimento` nulo (4.463) indica chamados abandonados ainda na espera, e `resolvido` nulo (2.014) são registros sem informação de resolução
* **Integridade `id_cliente`**: 1.330 chamados (~1%) referenciam clientes sem cadastro na `tb_clientes`. Não são sentinelas (IDs variados, cada um com um único chamado) — são dados sujos da origem. Decisão de tratamento (manter, sinalizar ou filtrar) fica para a camada Gold, no momento do join com a base de clientes

In [0]:
# === DEBUG: Qualidade restante - nulos por coluna e integridade id_cliente ===
df_silver = spark.table(f"{silver_schema}.tb_chamados")

print(f"Total de linhas: {df_silver.count():,}")

print("\nNulos por coluna:")
nulos = df_silver.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_silver.columns]).collect()[0]
for c in df_silver.columns:
    print(f"  {c}: {nulos[c]:,}")

# Integridade: clientes citados nos chamados x cadastro de clientes (bronze)
clientes = spark.table(f"{bronze_schema}.tb_clientes")
print(f"\nSchema tb_clientes: {clientes.columns}")
try:
    orfaos_cli = df_silver.select("id_cliente").distinct().join(
        clientes.select("id_cliente").distinct(), on="id_cliente", how="left_anti"
    )
    qtd_orfaos = orfaos_cli.count()
    print(f"Clientes citados nos chamados SEM cadastro: {qtd_orfaos}")
    if qtd_orfaos > 0:
        display(orfaos_cli.orderBy("id_cliente").limit(20))
        chamados_orfaos = df_silver.join(orfaos_cli, on="id_cliente", how="inner")
        print(f"Chamados ligados a clientes sem cadastro: {chamados_orfaos.count():,}")
except Exception as e:
    print(f"[ALERTA] Não foi possível cruzar com tb_clientes: {e}")

# Nulos por canal: identifica se os nulls de tempo são estruturais (ura/chatbot)
print("\nNulos por canal:")
display(
    df_silver.groupBy("canal")
      .agg(F.count("*").alias("chamados"),
           F.sum(F.col("tempo_espera").isNull().cast("int")).alias("espera_null"),
           F.sum(F.col("tempo_atendimento").isNull().cast("int")).alias("atendimento_null"),
           F.sum(F.col("resolvido").isNull().cast("int")).alias("resolvido_null"))
      .orderBy("canal")
)